<img src='http://hilpisch.com/tpq_logo.png' width="350px" align="right">

# Certificate Programs

**Mentoring Session**

## DX Analytics

The Python Quants GmbH

## Options Data

Retrieving the options data from the following CSV file:

    http://hilpisch.com/ref_eikon_option_data.csv  

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
import pandas as pd

In [ ]:
data = pd.read_csv('http://hilpisch.com/ref_eikon_option_data.csv',
                  parse_dates=['CF_DATE', 'EXPIR_DATE'], index_col=0)

In [ ]:
data.info()

## DX Analytics

Using the `dx` package (http://dx-analytics.com) to model and value the European put and call options in the file. In this context, making use of the implied volatility as provided in the file.

In [ ]:
import dx
import cufflinks
%matplotlib inline

In [ ]:
initial_value = data['CF_CLOSE'].iloc[0]
initial_value

In [ ]:
pricing_date = data['CF_DATE'].iloc[0].to_pydatetime()
pricing_date

In [ ]:
data.drop(0, inplace=True)

In [ ]:
# dx.constant_short_rate??

In [ ]:
r = dx.constant_short_rate('r', 0.0)

In [ ]:
me = dx.market_environment('dax', pricing_date)

In [ ]:
me.add_constant('initial_value', initial_value)
me.add_constant('volatility', data['IMP_VOLT'].iloc[0] / 100)
me.add_constant('final_date', data['EXPIR_DATE'].iloc[0].to_pydatetime())
me.add_constant('currency', 'EUR')
me.add_constant('frequency', 'D')
me.add_constant('paths', 50000)
me.add_curve('discount_curve', r)

In [ ]:
gbm = dx.geometric_brownian_motion('gbm', me)

In [ ]:
paths = pd.DataFrame(gbm.get_instrument_values(), index=gbm.time_grid)

In [ ]:
paths.info()

In [ ]:
pd.options.plotting.backend = "plotly"

In [ ]:
paths.iloc[:, :20].plot()

In [ ]:
meo = dx.market_environment('meo', me.pricing_date)
meo.add_environment(me)
meo.add_constant('maturity', data['EXPIR_DATE'].iloc[0].to_pydatetime())
meo.add_constant('strike', data['STRIKE_PRC'].iloc[0])

In [ ]:
meo.constants

In [ ]:
call = dx.valuation_mcs_european_single(
            name='call',
            underlying=gbm,
            mar_env=meo,
            payoff_func='np.maximum(maturity_value - strike, 0)')

In [ ]:
put = dx.valuation_mcs_european_single(
            name='put',
            underlying=gbm,
            mar_env=meo,
            payoff_func='np.maximum(strike - maturity_value, 0)')

In [ ]:
call.present_value()

In [ ]:
data.head(2)

In [ ]:
put.present_value()  # high due to high IV (203)

In [ ]:
%%time
data['VALUE'] = 0
for i, option in data.iterrows():
    if option['PUTCALLIND'] == 'CALL':
        payoff = 'np.maximum(maturity_value - strike, 0)'
    else:
        payoff = 'np.maximum(strike - maturity_value, 0)'
    me.add_constant('volatility', option['IMP_VOLT'] / 100)
    gbm = dx.geometric_brownian_motion('gbm', me)
    meo.add_constant('strike', option['STRIKE_PRC'])
    model = dx.valuation_mcs_european_single(
            name='eur_option', underlying=gbm,
            mar_env=meo, payoff_func=payoff)
    value = model.present_value(fixed_seed=True)
    data['VALUE'].iloc[i - 1] = value

In [ ]:
data.head(10)

In [ ]:
data.tail()

## Visualization

Comparing, visually, the model values and the market quotes for the European options.

In [ ]:
data.plot.scatter(x='CF_CLOSE', y='VALUE')

In [ ]:
data['DIFF'] = data['VALUE'] - data['CF_CLOSE']

In [ ]:
data['DIFF'].mean()

In [ ]:
data['DIFF%'] = data['DIFF'] / data['CF_CLOSE'] * 100

In [ ]:
data[data['CF_CLOSE'] > 0]['DIFF%'].mean()

In [ ]:
data[data['CF_CLOSE'] > 0].tail()

In [ ]:
data[data['CF_CLOSE'] > 50]['DIFF%'].plot(kind='hist')

In [ ]:
data[data['CF_CLOSE'] > 1000][['DIFF%', 'STRIKE_PRC']].plot(kind='bar', x='STRIKE_PRC')

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>